# 00_Configuracion

Configuración central del proyecto **accidentalidad-databricks**.

Este notebook define los parámetros compartidos del proyecto y crea el catálogo de Unity Catalog con los schemas de la arquitectura Medallion.

## Preparación reproducible
Crea catálogo, schemas Bronze/Silver/Gold/Quality y el Volume administrado `bronze.raw_files`.
Después cargue manualmente los cuatro CSV siguiendo `data/README.md`. Use un catálogo distinto
por entorno: el widget `entorno` es descriptivo y no proporciona aislamiento.
Se requieren privilegios para crear estos objetos en Unity Catalog.

In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.dropdown("entorno", "training", ["training", "dev", "demo"], "Entorno")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROY_ENTORNO = dbutils.widgets.get("entorno")
if PROY_ENTORNO not in {"training", "dev", "demo"}:
    raise ValueError("Entorno inválido")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {PROY_CATALOG} COMMENT 'Accidentalidad Barranquilla'")
for schema in ["bronze", "silver", "gold", "quality"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PROY_CATALOG}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {PROY_CATALOG}.bronze.raw_files")
RAW_PATH = f"/Volumes/{PROY_CATALOG}/bronze/raw_files"
print(f"Entorno: {PROY_ENTORNO}; cargar los cuatro CSV completos en {RAW_PATH}")

In [ ]:
# Verificación
display(spark.sql(f"SHOW SCHEMAS IN {PROY_CATALOG}"))